# Web 版聊天应用

预计阅读时间：10-15 分钟。

本节解读 [`07_WebChat.py`](07_WebChat.py)。它使用 Gradio 构建一个浏览器聊天界面，支持多轮对话、流式输出和 Markdown 展示。

## 运行前准备

Web 版除了 `openai` 和 `python-dotenv`，还需要安装 `gradio`：

In [2]:
!python3 -m pip install -q --upgrade "openai>=1.0" python-dotenv gradio

运行方式：
```bash
python3 07_WebChat.py
```

终端会输出本地网页地址，通常类似 `http://127.0.0.1:7860`。打开后即可聊天。

## 导入依赖

文件开头导入四类依赖：

In [3]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

- `os`：读取环境变量；
- `gradio`：创建 Web 聊天界面；
- `load_dotenv`：读取 `.env` 文件；
- `OpenAI`：使用 OpenAI 兼容 API 调用 GLM。

## 读取配置并创建客户端

这部分和前面章节一致：

In [4]:
load_dotenv()

api_key = os.getenv("ZAI_API_KEY")

client = OpenAI(
    api_key=api_key or "missing",
    base_url="https://open.bigmodel.cn/api/paas/v4/",
)
MODEL_NAME = "glm-5.2"

这里用 `api_key or "missing"` 是为了让程序即使没有配置 Key 也能启动页面；真正调用时会在 `chat()` 中提示用户先设置环境变量。

## system prompt

Web 版允许模型输出 Markdown，因此 system prompt 中明确写入这一点：

In [5]:
SYSTEM_PROMPT = (
    "你是一个通用学习助手。回答要简洁，注重原理阐释和示例引导；"
    "优先通过苏格拉底式提问启发用户思考。可以使用 Markdown。"
)

这样模型更可能返回标题、列表、代码块等适合 Web 展示的内容。

## 转换历史消息

Gradio 会把历史对话传给 `chat(message, history)`。不同版本的 Gradio 历史格式可能不同，因此 `to_messages()` 同时处理两种情况：

In [6]:
def to_messages(message: str, history: list) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in history:
        if isinstance(item, dict):
            messages.append({"role": item["role"], "content": item["content"]})
        else:
            user, assistant = item
            messages.append({"role": "user", "content": user})
            messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    return messages

逻辑说明：

- 先放入 `system` 消息；
- 如果历史项已经是字典，就直接转成 API 格式；
- 如果历史项是 `(user, assistant)` 元组，就拆成两条消息；
- 最后追加当前用户输入。

这样无论 Gradio 版本如何变化，发送给模型的都是统一的 `messages`。

## 流式聊天函数

`chat()` 是 Gradio 调用的核心函数：

In [7]:
def chat(message: str, history: list):
    if not api_key:
        yield "请先设置环境变量 `ZAI_API_KEY`。"
        return

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=to_messages(message, history),
        temperature=0.6,
        stream=True,
    )

    answer = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        answer += delta
        yield answer

这里有两个关键点：

- `stream=True`：让模型边生成边返回；
- `yield answer`：每次把当前累计答案交给 Gradio，页面就能实时更新。

如果没有 API Key，函数先 `yield` 一条提示，然后 `return` 结束。

## 创建 Web 界面

最后用 `gr.ChatInterface` 创建聊天页面：

In [8]:
demo = gr.ChatInterface(
    fn=chat,
    title="GLM Chat Demo",
    description="支持多轮对话、流式输出和 Markdown 展示。",
)

`fn=chat` 表示：用户每次在页面输入内容后，Gradio 会调用 `chat()`。Gradio 的聊天组件会自动展示历史，也会把 Markdown 渲染成更易读的格式。

## 程序入口

最后启动 Web 应用：

In [9]:
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/sunq/anaconda3/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


直接运行 `python3 07_WebChat.py` 时，`demo.launch()` 会启动本地 Web 服务。

## 和`06_命令行聊天应用`的关系

Web 版和命令行版的核心相同：

- 都使用 OpenAI 兼容 API；
- 都需要 system prompt；
- 都要把历史对话整理成 `messages`；
- 都调用 `client.chat.completions.create()`。

区别是：06 用 `while` 循环和 `print()` 交互；07 用 Gradio 接管输入框、历史显示和页面渲染。

## 更多

[`08_AppChat.py`](08_AppChat.py) 是一个 PySide6 桌面应用示例。它不依赖浏览器，而是使用桌面窗口、后台线程和信号机制实现流式输出，供感兴趣的同学继续阅读。

## 练习

1. 尝试修改 `SYSTEM_PROMPT`，让 Web 应用更偏向“英文写作润色助手”。要求它以批改的形式润色英文输入，再用markdown展示修改内容。

2. 尝试修改 `SYSTEM_PROMPT`，让 Web 应用更偏向“英文写作润色助手”。要求它以批改的形式润色英文输入，再用markdown展示修改内容。